In [ ]:
import sqlite3
from openai import OpenAI 
from dotenv import load_dotenv
import gradio as gr
import os 
import json


DB = "price.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")


In [ ]:


load_dotenv(override=True)

openai_api_key = os.getenv('OPEN_API_KEY')
if not openai_api_key:
    print("OPEN_API_KEY not available")
else:
    print("OPEN_API_KEY has set up")

In [ ]:
def set_ticket_price(city: str, price: float):
    """
        Set ticket price for certain city
    """
    with sqlite3.Connection(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?", (city.lower(), price, price))
        conn.commit()
        return f"The price for {city} is {price}"



In [ ]:

def get_ticket_price(city: str):
    """
        get ticket price by city
    """
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(),))
        res = cursor.fetchone()
        return f"The price for {city} is {res[0]}" if res else f"There is not price available for {city}"



In [ ]:
# construct set ticket schema
set_ticket_price_function = {
    "name": "set_ticket_price",
    "description": "set ticket price for a destination",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "the destionation city that customer want to travel"
            },
            "price": {
                "type": "number",
                "description": "The price that need paid for the travel"
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False,
    },
}




In [ ]:
# construct get ticket price schema
get_ticket_price_function = {
    "name": "get_ticket_price",
    "description": "get ticket price for a destination",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destionation city that customer want to travel"
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

In [ ]:
# construct tools for openai API call
tools = [
    {"type": "function", "function": get_ticket_price_function},
    {"type": "function", "function": set_ticket_price_function}
]

In [ ]:
system_message=""" 
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answer, no more than 1 sentences.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
openai = OpenAI()
MODEL="gpt-4.1-mini"

In [ ]:
def handle_tool_calls(message):
    """
    handl tool for get and set price for desitional
    """
    respones = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")
        price = arguments.get("price")
        function_name = tool_call.function.name
        if function_name == "get_ticket_price":
            price_details = get_ticket_price(city)
            respones.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

        if function_name == "set_ticket_price":

            respones.append({
                "role": "tool",
                "content": set_ticket_price(city, price),
                "tool_call_id": tool_call.id
            })

    return respones

In [ ]:
def chat(message, history):
    """
        set the chat function for gradio callback
    """
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # construct message list with system message, history , and current message
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # invoke openai call
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # when tool calls existed
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        # call tools to generate response
        response = handle_tool_calls(message)
        # append message and response
        messages.append(message)
        messages.extend(response)
        # call openai again to generate openai response
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools) 


    # return final response
        
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()